In [3]:
!pip install pyspark findspark

In [4]:
import findspark
findspark.init()

In [5]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, max, min, count, round, explode, split

In [11]:
spark = SparkSession.builder.appName("CovidAnalysis").getOrCreate()
print("Spark session started successfully")

Spark session started successfully


In [ ]:
country_wise_df = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://localhost:9000/user/covid/country_wise_latest.csv")
full_grouped_df = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://localhost:9000/user/covid/full_grouped.csv")
day_wise_df = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://localhost:9000/user/covid/day_wise.csv")
worldometer_df = spark.read.option("header", True).option("inferSchema", True).csv("hdfs://localhost:9000/user/covid/worldometer_data.csv")

print("country wise Schema: ")
country_wise_df.printSchema()

print("worldometer Schema: ")
worldometer_df.printSchema()

In [29]:
from pyspark.sql.functions import sum

regional_summary = country_wise_df.groupBy("Country/Region").agg(
    sum("Confirmed").alias("Total_Confirmed"),
    sum("Recovered").alias("Total_Recovered"),
    sum("Deaths").alias("Total_Deaths")
)
regional_summary.orderBy("Total_Confirmed", ascending=False).show(10)


+--------------+---------------+---------------+------------+
|Country/Region|Total_Confirmed|Total_Recovered|Total_Deaths|
+--------------+---------------+---------------+------------+
|            US|        4290259|        1325804|      148011|
|        Brazil|        2442375|        1846641|       87618|
|         India|        1480073|         951166|       33408|
|        Russia|         816680|         602249|       13334|
|  South Africa|         452529|         274925|        7067|
|        Mexico|         395489|         303810|       44022|
|          Peru|         389717|         272547|       18418|
|         Chile|         347923|         319954|        9187|
|United Kingdom|         301708|           1437|       45844|
|          Iran|         293606|         255144|       15912|
+--------------+---------------+---------------+------------+
only showing top 10 rows



In [31]:
from pyspark.sql.functions import col, round

recovery_rate_df = country_wise_df.withColumn(
    "Recovery Rate", round((col("Recovered") / col("Confirmed")) * 100, 2)
).select("Country/Region", "Recovery Rate")

recovery_rate_df.orderBy("Recovery Rate", ascending=False).show(10)


+--------------+-------------+
|Country/Region|Recovery Rate|
+--------------+-------------+
|      Holy See|        100.0|
|       Grenada|        100.0|
|      Dominica|        100.0|
|      Djibouti|        98.38|
|       Iceland|        98.33|
|        Brunei|        97.87|
|   New Zealand|        97.24|
|         Qatar|        97.02|
|      Malaysia|         96.6|
|     Mauritius|        96.51|
+--------------+-------------+
only showing top 10 rows



In [33]:
comparison_df = country_wise_df.withColumn(
    "Recovery Rate", round((col("Recovered") / col("Confirmed")) * 100, 2)
).withColumn(
    "Fatality Rate", round((col("Deaths") / col("Confirmed")) * 100, 2)
).select("Country/Region", "Recovery Rate", "Fatality Rate")

comparison_df.orderBy("Country/Region").show(10)


+-------------------+-------------+-------------+
|     Country/Region|Recovery Rate|Fatality Rate|
+-------------------+-------------+-------------+
|        Afghanistan|        69.49|          3.5|
|            Albania|        56.25|         2.95|
|            Algeria|        67.34|         4.16|
|            Andorra|        88.53|         5.73|
|             Angola|        25.47|         4.32|
|Antigua and Barbuda|        75.58|         3.49|
|          Argentina|        43.35|         1.83|
|            Armenia|        71.32|          1.9|
|          Australia|        60.84|         1.09|
|            Austria|        88.75|         3.47|
+-------------------+-------------+-------------+
only showing top 10 rows



In [35]:
country_wise_df.select("Country/Region", "Deaths").orderBy("Deaths", ascending=True).show(10)


+--------------+------+
|Country/Region|Deaths|
+--------------+------+
|      Dominica|     0|
|        Bhutan|     0|
|      Cambodia|     0|
|       Eritrea|     0|
|          Fiji|     0|
|     Greenland|     0|
|       Grenada|     0|
|      Holy See|     0|
|          Laos|     0|
|      Mongolia|     0|
+--------------+------+
only showing top 10 rows



In [37]:
country_wise_df.select("Country/Region", "Confirmed").orderBy("Confirmed", ascending=False).show(10)


+--------------+---------+
|Country/Region|Confirmed|
+--------------+---------+
|            US|  4290259|
|        Brazil|  2442375|
|         India|  1480073|
|        Russia|   816680|
|  South Africa|   452529|
|        Mexico|   395489|
|          Peru|   389717|
|         Chile|   347923|
|United Kingdom|   301708|
|          Iran|   293606|
+--------------+---------+
only showing top 10 rows



In [39]:
global_recovery = country_wise_df.select(
    round((sum("Recovered") / sum("Confirmed")) * 100, 2).alias("Global Recovery Rate")
)

global_recovery.show()


+--------------------+
|Global Recovery Rate|
+--------------------+
|               57.45|
+--------------------+



In [43]:
trend_df = full_grouped_df.groupBy("Date", "WHO Region").agg(
    sum("Confirmed").alias("Total_Confirmed"),
    sum("Deaths").alias("Total_Deaths"),
    sum("Recovered").alias("Total_Recovered")
)

trend_df.orderBy("Date", "WHO Region").show(20)



+----------+--------------------+---------------+------------+---------------+
|      Date|          WHO Region|Total_Confirmed|Total_Deaths|Total_Recovered|
+----------+--------------------+---------------+------------+---------------+
|2020-01-22|              Africa|              0|           0|              0|
|2020-01-22|            Americas|              1|           0|              0|
|2020-01-22|Eastern Mediterra...|              0|           0|              0|
|2020-01-22|              Europe|              0|           0|              0|
|2020-01-22|     South-East Asia|              2|           0|              0|
|2020-01-22|     Western Pacific|            552|          17|             28|
|2020-01-23|              Africa|              0|           0|              0|
|2020-01-23|            Americas|              1|           0|              0|
|2020-01-23|Eastern Mediterra...|              0|           0|              0|
|2020-01-23|              Europe|              0|   

In [47]:
continent_cases = country_wise_df.groupBy("WHO Region").agg(
    sum("Confirmed").alias("Total_Cases")
)

continent_cases.orderBy("Total_Cases").show(1)

+---------------+-----------+
|     WHO Region|Total_Cases|
+---------------+-----------+
|Western Pacific|     292428|
+---------------+-----------+
only showing top 1 row



In [51]:
recovery_continent = country_wise_df.groupBy("WHO Region").agg(
    round((sum("Recovered") / sum("Confirmed")) * 100, 2).alias("Recovery Rate")
)

recovery_continent.orderBy("Recovery Rate", ascending=False).show()

+--------------------+-------------+
|          WHO Region|Recovery Rate|
+--------------------+-------------+
|Eastern Mediterra...|        80.59|
|     Western Pacific|        70.71|
|     South-East Asia|        63.04|
|              Africa|        60.93|
|              Europe|        60.42|
|            Americas|        50.55|
+--------------------+-------------+



In [58]:

global_death_percent = country_wise_df.select(
    (round((sum("Deaths") / sum("Confirmed")) * 100, 2)).alias("Global Death Percentage")
)
global_death_percent.show()

local_death_percent = country_wise_df.withColumn(
    "Local Death Percentage", round((col("Deaths") / col("Confirmed")) * 100, 2)
).select("Country/Region", "Local Death Percentage")
local_death_percent.orderBy("Local Death Percentage", ascending=False).show(10)


+-----------------------+
|Global Death Percentage|
+-----------------------+
|                   3.97|
+-----------------------+

+--------------+----------------------+
|Country/Region|Local Death Percentage|
+--------------+----------------------+
|         Yemen|                 28.56|
|United Kingdom|                 15.19|
|       Belgium|                 14.79|
|         Italy|                 14.26|
|        France|                 13.71|
|       Hungary|                  13.4|
|   Netherlands|                 11.53|
|        Mexico|                 11.13|
|         Spain|                 10.44|
|Western Sahara|                  10.0|
+--------------+----------------------+
only showing top 10 rows

